# EveryQuery vs EIC: Multi-Duration Results

Single analysis notebook for the updated preprint. Loads EQ multi-duration eval,
EIC baseline, and prevalence data, then runs verification, statistical comparisons,
loss characterization, and generates all figures inline.

## Setup and Data Loading

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import polars as pl
from scipy import stats as sp_stats

matplotlib.rcParams.update({
    "font.size": 8,
    "font.family": "sans-serif",
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

%matplotlib inline

In [ ]:
REPO = Path(".").resolve().parent
DATA = REPO / "data"

EQ_MODEL_DIR = DATA / "outputs" / "2026-04-02" / "23-43-54"
EQ_PARQUET = EQ_MODEL_DIR / "eval" / "best" / "eval_aucs_held_out_20260410_192213.parquet"
EIC_PARQUET = DATA / "baselines" / "eic" / "temporal_auc_results_10000_ID__8db2be6fadf8.parquet"
PREVALENCE_CSV = DATA / "held_out_prevalence_7573f855c4b050a9d79d57fefd8a139c.csv"

DURATIONS = [30, 90, 180, 365, 730]
TRAINED_DURATIONS = {30, 180}
OOD_DURATIONS = {90, 365, 730}
INTERP_OOD = {90}
EXTRAP_OOD = {365, 730}

In [ ]:
# Load EQ
eq_raw = pl.read_parquet(EQ_PARQUET)
eq_raw = eq_raw.with_columns(
    pl.when(pl.col("duration_days") == 731)
    .then(730)
    .otherwise(pl.col("duration_days"))
    .alias("duration_days")
)
print(f"EQ raw: {eq_raw.shape}")

# Load EIC (wide -> long)
eic_wide = pl.read_parquet(EIC_PARQUET)
eic_wide = eic_wide.with_columns(
    (pl.col("duration").dt.total_microseconds() / (1e6 * 86400))
    .cast(pl.Int64)
    .alias("duration_days")
)
auc_cols = [c for c in eic_wide.columns if c.startswith("AUC/")]
eic_raw = eic_wide.unpivot(
    on=auc_cols,
    index=["duration_days"],
    variable_name="code_slug_raw",
    value_name="eic_auc",
).with_columns(
    pl.col("code_slug_raw").str.strip_prefix("AUC/").alias("code_slug")
).drop("code_slug_raw")
print(f"EIC unpivoted: {eic_raw.shape}")

# Load prevalence
prev_raw = pl.read_csv(PREVALENCE_CSV)
prev_raw = prev_raw.with_columns(
    pl.when(pl.col("duration_days") == 731)
    .then(730)
    .otherwise(pl.col("duration_days"))
    .alias("duration_days")
)
print(f"Prevalence raw: {prev_raw.shape}")

In [ ]:
# Inner join EQ + EIC
joined_raw = eq_raw.join(eic_raw, on=["code_slug", "duration_days"], how="inner")
print(f"After join: {joined_raw.shape[0]} rows")

# Filter nulls, compute delta
joined = (
    joined_raw
    .filter(pl.col("occurs_auc").is_not_null() & pl.col("eic_auc").is_not_null())
    .with_columns(
        (pl.col("occurs_auc") - pl.col("eic_auc")).alias("delta_auc")
    )
)
print(f"After null filter: {len(joined)} rows")

# Join prevalence
df = joined.join(prev_raw, on=["code", "duration_days"], how="inner")
print(f"After prevalence join: {len(df)} rows")

## Data Verification

In [ ]:
print("Row counts at each stage:")
print(f"  EQ raw:             {eq_raw.shape[0]}")
print(f"  EIC unpivoted:      {eic_raw.shape[0]}")
print(f"  After inner join:   {joined_raw.shape[0]}")
print(f"  After null filter:  {len(joined)}")
print(f"  After prev join:    {len(df)}")

In [ ]:
# Code slug overlap
eq_slugs = set(eq_raw["code_slug"].unique().to_list())
eic_slugs = set(eic_raw["code_slug"].unique().to_list())
print(f"EQ codes: {len(eq_slugs)}, EIC codes: {len(eic_slugs)}")
print(f"EQ codes not in EIC: {eq_slugs - eic_slugs}")
print(f"EIC codes not in EQ: {eic_slugs - eq_slugs}")

# Duration coverage
eq_durs = sorted(eq_raw["duration_days"].unique().to_list())
eic_durs = sorted(eic_raw["duration_days"].unique().to_list())
print(f"\nEQ durations:  {eq_durs}")
print(f"EIC durations: {eic_durs}")

In [ ]:
# Null audit on joined (pre-prevalence) frame
print("Null counts per column (joined frame):")
for col in joined.columns:
    nc = joined[col].null_count()
    if nc > 0:
        print(f"  {col}: {nc}")
if all(joined[c].null_count() == 0 for c in joined.columns):
    print("  (none)")

# Bucket distribution
print(f"\nBucket distribution:")
joined["bucket"].value_counts().sort("bucket")

In [ ]:
# Summary stats
summary_cols = ["occurs_auc", "eic_auc", "delta_auc"]
print("Overall summary:")
joined.select(
    *[
        expr
        for col in summary_cols
        for expr in (
            pl.col(col).mean().alias(f"{col}_mean"),
            pl.col(col).median().alias(f"{col}_median"),
            pl.col(col).std().alias(f"{col}_std"),
        )
    ]
)

In [ ]:
print("Per-duration summary:")
joined.group_by("duration_days").agg(
    *[
        expr
        for col in summary_cols
        for expr in (
            pl.col(col).mean().alias(f"{col}_mean"),
            pl.col(col).median().alias(f"{col}_median"),
            pl.col(col).std().alias(f"{col}_std"),
        )
    ]
).sort("duration_days")

## EQ vs EIC Comparison

In [ ]:
def bootstrap_ci(deltas: np.ndarray, n_boot: int = 10000, alpha: float = 0.05, seed: int = 42):
    rng = np.random.default_rng(seed)
    means = np.array([
        deltas[rng.integers(0, len(deltas), size=len(deltas))].mean()
        for _ in range(n_boot)
    ])
    lo = np.percentile(means, 100 * alpha / 2)
    hi = np.percentile(means, 100 * (1 - alpha / 2))
    return lo, hi


def compute_metrics(sub: pl.DataFrame):
    deltas = sub["delta_auc"].to_numpy()
    n = len(deltas)
    wins = int((deltas > 0).sum())
    mean_d = float(deltas.mean())
    median_d = float(np.median(deltas))
    lo, hi = bootstrap_ci(deltas)
    if n >= 10:
        _, p = sp_stats.wilcoxon(
            sub["occurs_auc"].to_numpy(),
            sub["eic_auc"].to_numpy(),
            alternative="two-sided",
        )
    else:
        p = float("nan")
    return {
        "N_pairs": n,
        "EQ_wins": wins,
        "Win%": round(100 * wins / n, 1),
        "Mean_dAUC": round(mean_d, 4),
        "Median_dAUC": round(median_d, 4),
        "95%_CI": f"[{lo:.4f}, {hi:.4f}]",
        "Wilcoxon_p": p,
    }

In [ ]:
# Part A: Overall metrics
m = compute_metrics(joined)
print("Overall EQ vs EIC comparison")
print(f"  N paired tasks x durations : {m['N_pairs']}")
print(f"  EQ wins                    : {m['EQ_wins']} / {m['N_pairs']}  ({m['Win%']}%)")
print(f"  Mean delta AUC (EQ - EIC)  : {m['Mean_dAUC']}")
print(f"  Median delta AUC           : {m['Median_dAUC']}")
print(f"  95% bootstrap CI (mean)    : {m['95%_CI']}")
print(f"  Wilcoxon signed-rank p     : {m['Wilcoxon_p']:.4e}")

In [ ]:
# Part B: Per-duration breakdown
rows_b = []
for dur in DURATIONS:
    sub = joined.filter(pl.col("duration_days") == dur)
    if len(sub) == 0:
        continue
    met = compute_metrics(sub)
    rows_b.append({
        "Duration": dur,
        "Trained": "ID" if dur in TRAINED_DURATIONS else "OOD",
        **met,
    })

pl.DataFrame(rows_b)

In [ ]:
# Part C: Per-duration x bucket breakdown
rows_c = []
for dur in DURATIONS:
    for bucket in ["id", "ood"]:
        sub = joined.filter(
            (pl.col("duration_days") == dur) & (pl.col("bucket") == bucket)
        )
        if len(sub) == 0:
            continue
        met = compute_metrics(sub)
        rows_c.append({
            "Duration": dur,
            "Trained": "ID" if dur in TRAINED_DURATIONS else "OOD",
            "Bucket": bucket,
            **met,
        })

pl.DataFrame(rows_c)

## Loss Characterization

In [ ]:
# All losses: (code, duration) pairs where EIC > EQ
losses = joined.filter(pl.col("delta_auc") < 0).sort("delta_auc")
print(f"Total losses: {len(losses)}")
print()

# Concentration: which codes lose and at which durations
loss_codes = losses.group_by("code_slug", "bucket").agg(
    pl.len().alias("n_losses"),
    pl.col("duration_days").sort().alias("losing_durations"),
).sort("n_losses", descending=True)

print(f"Unique codes that lose at least once: {len(loss_codes)}")
print()
loss_codes

In [ ]:
# Win/loss matrix: 40 codes x 5 durations
all_slugs = sorted(joined["code_slug"].unique().to_list())

matrix_rows = []
for slug in all_slugs:
    sub = joined.filter(pl.col("code_slug") == slug)
    bucket = sub["bucket"][0]
    row_data = {"code_slug": slug, "bucket": bucket}
    n_losses = 0
    for dur in DURATIONS:
        match = sub.filter(pl.col("duration_days") == dur)
        if len(match) == 0:
            row_data[str(dur)] = "-"
        elif match["delta_auc"][0] > 0:
            row_data[str(dur)] = "W"
        elif match["delta_auc"][0] < 0:
            row_data[str(dur)] = "L"
            n_losses += 1
        else:
            row_data[str(dur)] = "T"
    row_data["n_losses"] = n_losses
    matrix_rows.append(row_data)

matrix_rows.sort(key=lambda r: (-r["n_losses"], r["code_slug"]))

# Concentration summary
for n in range(1, 6):
    cnt = sum(1 for r in matrix_rows if r["n_losses"] == n)
    if cnt > 0:
        print(f"Codes losing at {n} duration(s): {cnt}")
print()

pl.DataFrame(matrix_rows)

In [ ]:
# Win/loss pattern analysis
patterns = Counter()
for r in matrix_rows:
    pattern = tuple(r[str(d)] for d in DURATIONS)
    patterns[pattern] += 1

print("Most common win/loss patterns (30, 90, 180, 365, 730):")
for pat, cnt in patterns.most_common(10):
    print(f"  {pat}: {cnt} codes")

In [ ]:
# Duration generalization tests
deltas_id = joined.filter(pl.col("duration_days").is_in(list(TRAINED_DURATIONS)))["delta_auc"].to_numpy()
deltas_ood = joined.filter(pl.col("duration_days").is_in(list(OOD_DURATIONS)))["delta_auc"].to_numpy()

print(f"ID durations (30, 180):     N={len(deltas_id):>3}, mean={deltas_id.mean():.4f}, median={np.median(deltas_id):.4f}")
print(f"OOD durations (90,365,730): N={len(deltas_ood):>3}, mean={deltas_ood.mean():.4f}, median={np.median(deltas_ood):.4f}")

stat_mw, p_mw = sp_stats.mannwhitneyu(deltas_id, deltas_ood, alternative="two-sided")
print(f"\nMann-Whitney U: U={stat_mw:.1f}, p={p_mw:.4f}")

# Finer split
deltas_interp = joined.filter(pl.col("duration_days").is_in(list(INTERP_OOD)))["delta_auc"].to_numpy()
deltas_extrap = joined.filter(pl.col("duration_days").is_in(list(EXTRAP_OOD)))["delta_auc"].to_numpy()

print(f"\nID (30, 180):          N={len(deltas_id):>3}, mean={deltas_id.mean():.4f}")
print(f"Interp OOD (90):       N={len(deltas_interp):>3}, mean={deltas_interp.mean():.4f}")
print(f"Extrap OOD (365, 730): N={len(deltas_extrap):>3}, mean={deltas_extrap.mean():.4f}")

stat_kw, p_kw = sp_stats.kruskal(deltas_id, deltas_interp, deltas_extrap)
print(f"\nKruskal-Wallis (3 groups): H={stat_kw:.3f}, p={p_kw:.4f}")

## Figures

In [ ]:
# Figure A: Per-duration delta AUC bar chart
fig, ax = plt.subplots(figsize=(3.5, 2.8))

means, los, his, win_rates, colors_list = [], [], [], [], []
color_id = "#4878CF"
color_ood = "#E1812C"

for dur in DURATIONS:
    sub = joined.filter(pl.col("duration_days") == dur)
    deltas = sub["delta_auc"].to_numpy()
    m = deltas.mean()
    lo, hi = bootstrap_ci(deltas)
    wr = 100 * (deltas > 0).sum() / len(deltas)
    means.append(m)
    los.append(m - lo)
    his.append(hi - m)
    win_rates.append(wr)
    colors_list.append(color_id if dur in TRAINED_DURATIONS else color_ood)

x = np.arange(len(DURATIONS))
bars = ax.bar(
    x, means, yerr=[los, his], capsize=3, color=colors_list,
    edgecolor="white", linewidth=0.5, width=0.65, error_kw={"linewidth": 0.8},
)

for i, (xi, m_val, wr) in enumerate(zip(x, means, win_rates)):
    ax.text(xi, m_val + his[i] + 0.006, f"{wr:.0f}%",
            ha="center", va="bottom", fontsize=7, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels([f"{d}d" for d in DURATIONS])
ax.set_xlabel("Prediction horizon (days)")
ax.set_ylabel("Mean $\\Delta$AUC (EQ $-$ EIC)")
ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax.set_ylim(0, max(means) + max(his) + 0.055)

legend_elements = [
    plt.Rectangle((0, 0), 1, 1, facecolor=color_id, edgecolor="none", label="Trained duration"),
    plt.Rectangle((0, 0), 1, 1, facecolor=color_ood, edgecolor="none", label="Untrained duration"),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=7, loc="upper left",
          bbox_to_anchor=(0.0, 1.0))

fig.tight_layout()
plt.show()

In [ ]:
# Figure B: Task-level scatter (delta AUC by code, colored by duration)
code_order = (
    joined.group_by("code_slug")
    .agg(pl.col("delta_auc").mean().alias("mean_delta"))
    .sort("mean_delta")
)
code_list = code_order["code_slug"].to_list()
code_to_x = {c: i for i, c in enumerate(code_list)}

fig, ax = plt.subplots(figsize=(7, 3.2))

ax.axhspan(-0.6, 0, color="#FDECEC", zorder=0)

dur_colors = {30: "#2166AC", 90: "#4393C3", 180: "#92C5DE", 365: "#F4A582", 730: "#D6604D"}
dur_markers = {30: "o", 90: "s", 180: "^", 365: "D", 730: "v"}

for dur in DURATIONS:
    sub = joined.filter(pl.col("duration_days") == dur)
    xs = [code_to_x[c] for c in sub["code_slug"].to_list()]
    ys = sub["delta_auc"].to_numpy()
    ax.scatter(
        xs, ys, c=dur_colors[dur], marker=dur_markers[dur], s=22, alpha=0.85,
        edgecolors="white", linewidths=0.3, label=f"{dur}d", zorder=3,
    )

ax.axhline(0, color="#555555", linewidth=0.7, linestyle="-", zorder=2)
ax.set_xlabel("Clinical task (sorted by mean $\\Delta$AUC)")
ax.set_ylabel("$\\Delta$AUC (EQ $-$ EIC)")
ax.set_xticks([])
ax.legend(frameon=False, fontsize=7, ncol=5, loc="upper left",
          handletextpad=0.3, columnspacing=0.8)

fig.tight_layout()
plt.show()

In [ ]:
# Figure C: Heatmap dual panel (EQ AUC + delta AUC)
eq_for_heat = joined.select("code_slug", "duration_days", "occurs_auc", "delta_auc", "bucket")

code_sort = (
    eq_for_heat.group_by("code_slug")
    .agg(pl.col("occurs_auc").mean().alias("mean_eq_auc"))
    .sort("mean_eq_auc", descending=True)
)
sorted_codes = code_sort["code_slug"].to_list()

n_codes = len(sorted_codes)
n_durs = len(DURATIONS)
eq_matrix = np.full((n_codes, n_durs), np.nan)
delta_matrix = np.full((n_codes, n_durs), np.nan)
bucket_labels = []

for i, slug in enumerate(sorted_codes):
    sub = eq_for_heat.filter(pl.col("code_slug") == slug)
    bucket_labels.append(sub["bucket"][0])
    for j, dur in enumerate(DURATIONS):
        row = sub.filter(pl.col("duration_days") == dur)
        if len(row) > 0:
            eq_matrix[i, j] = row["occurs_auc"][0]
            delta_matrix[i, j] = row["delta_auc"][0]


def short_label(slug, bkt):
    parts = slug.rsplit("__", 1)
    name = parts[0][:22]
    tag = "id" if bkt == "id" else "ood"
    return f"{name} [{tag}]"


y_labels = [short_label(s, b) for s, b in zip(sorted_codes, bucket_labels)]

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(7, 11),
    gridspec_kw={"width_ratios": [1, 1], "wspace": 0.05},
)

cmap = "RdYlGn"
vmin = min(np.nanmin(eq_matrix), np.nanmin(delta_matrix))
vmax = max(np.nanmax(eq_matrix), np.nanmax(delta_matrix))

im1 = ax1.imshow(eq_matrix, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
ax1.set_xticks(range(n_durs))
ax1.set_xticklabels([f"{d}d" for d in DURATIONS], fontsize=7)
ax1.set_yticks(range(n_codes))
ax1.set_yticklabels(y_labels, fontsize=5.5)
ax1.set_title("EQ AUC", fontsize=9, fontweight="bold", pad=8)
ax1.set_xlabel("Prediction horizon")

im2 = ax2.imshow(delta_matrix, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
ax2.set_xticks(range(n_durs))
ax2.set_xticklabels([f"{d}d" for d in DURATIONS], fontsize=7)
ax2.set_yticks(range(n_codes))
ax2.set_yticklabels([], fontsize=5.5)
ax2.set_title("$\\Delta$AUC (EQ $-$ EIC)", fontsize=9, fontweight="bold", pad=8)
ax2.set_xlabel("Prediction horizon")

cb = fig.colorbar(im1, ax=[ax1, ax2], shrink=0.35, pad=0.08, aspect=25, location="bottom")
cb.ax.tick_params(labelsize=7)
cb.set_label("AUC", fontsize=8)

plt.show()

In [ ]:
# Prevalence scatter
dur_colors_prev = {30: "#2166AC", 90: "#4393C3", 180: "#92C5DE", 365: "#F4A582", 730: "#D6604D"}
bucket_markers = {"id": "o", "ood": "s"}

# Pooled Spearman for annotation
rho_delta, p_delta = sp_stats.spearmanr(df["prevalence"].to_numpy(), df["delta_auc"].to_numpy())

fig, ax = plt.subplots(figsize=(5, 3.5))

for dur in DURATIONS:
    for bkt, marker in bucket_markers.items():
        sub = df.filter((pl.col("duration_days") == dur) & (pl.col("bucket") == bkt))
        if len(sub) == 0:
            continue
        ax.scatter(
            sub["prevalence"].to_numpy(), sub["delta_auc"].to_numpy(),
            c=dur_colors_prev[dur], marker=marker, s=22, alpha=0.8,
            edgecolors="white", linewidths=0.3, zorder=3,
        )

legend_dur = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=dur_colors_prev[d],
           markersize=5, label=f"{d}d")
    for d in DURATIONS
]
legend_bkt = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=5, label="ID"),
    Line2D([0], [0], marker="s", color="w", markerfacecolor="gray", markersize=5, label="OOD"),
]

ax.axhline(0, color="#555555", linewidth=0.7, linestyle="-", zorder=1)
ax.set_xscale("log")
ax.set_xlabel("Prevalence (log scale)")
ax.set_ylabel("$\\Delta$AUC (EQ $-$ EIC)")

ax.text(
    0.97, 0.05, f"Spearman $\\rho$ = {rho_delta:+.2f}\n(p = {p_delta:.1e})",
    transform=ax.transAxes, ha="right", va="bottom", fontsize=7,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#cccccc", alpha=0.9),
)

leg1 = ax.legend(
    handles=legend_dur, frameon=False, fontsize=6.5, loc="upper left",
    title="Duration", title_fontsize=7, handletextpad=0.3,
)
ax.add_artist(leg1)
ax.legend(
    handles=legend_bkt, frameon=False, fontsize=6.5, loc="lower left",
    title="Code bucket", title_fontsize=7, handletextpad=0.3,
)

fig.tight_layout()
plt.show()

In [ ]:
# Prevalence trajectories
code_traj = (
    df.filter(pl.col("duration_days").is_in([30, 730]))
    .group_by("code_slug")
    .agg(
        pl.col("prevalence").filter(pl.col("duration_days") == 30).first().alias("prev_30"),
        pl.col("prevalence").filter(pl.col("duration_days") == 730).first().alias("prev_730"),
        pl.col("delta_auc").filter(pl.col("duration_days") == 30).first().alias("delta_30"),
        pl.col("delta_auc").filter(pl.col("duration_days") == 730).first().alias("delta_730"),
        pl.col("bucket").first().alias("bucket"),
    )
    .filter(pl.col("prev_30").is_not_null() & pl.col("prev_730").is_not_null())
    .with_columns(
        (pl.col("prev_730") / pl.col("prev_30")).alias("prev_ratio"),
        (pl.col("delta_730") - pl.col("delta_30")).alias("delta_change"),
    )
    .sort("prev_ratio", descending=True)
)

ratios = code_traj["prev_ratio"].to_numpy()
delta_changes = code_traj["delta_change"].to_numpy()
rho_traj, p_traj = sp_stats.spearmanr(ratios, delta_changes)

fig, ax = plt.subplots(figsize=(5, 3.5))

# All codes as faint background
all_codes = df["code_slug"].unique().to_list()
for slug in all_codes:
    sub = df.filter(pl.col("code_slug") == slug).sort("duration_days")
    if len(sub) < 2:
        continue
    ax.plot(
        sub["prevalence"].to_numpy(), sub["delta_auc"].to_numpy(),
        color="#cccccc", linewidth=0.4, alpha=0.5, zorder=1,
    )

# Highlight top-10 codes with biggest prevalence shift
top_codes = code_traj.head(10)["code_slug"].to_list()
cmap_lines = plt.cm.viridis(np.linspace(0.2, 0.9, len(top_codes)))
for i, slug in enumerate(top_codes):
    sub = df.filter(pl.col("code_slug") == slug).sort("duration_days")
    ax.plot(
        sub["prevalence"].to_numpy(), sub["delta_auc"].to_numpy(),
        color=cmap_lines[i], linewidth=1.2, alpha=0.9, zorder=3, marker="o", markersize=3,
    )

ax.axhline(0, color="#555555", linewidth=0.7, linestyle="-", zorder=2)
ax.set_xscale("log")
ax.set_xlabel("Prevalence (log scale)")
ax.set_ylabel("$\\Delta$AUC (EQ $-$ EIC)")
ax.set_title("Code trajectories across durations", fontsize=9)
ax.text(
    0.97, 0.05, f"$\\rho$(prev ratio, $\\Delta$gap) = {rho_traj:+.2f}",
    transform=ax.transAxes, ha="right", va="bottom", fontsize=7,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#cccccc", alpha=0.9),
)

fig.tight_layout()
plt.show()

## Per-Duration Spearman Correlations

In [ ]:
per_dur_rows = []
for dur in DURATIONS:
    sub = df.filter(pl.col("duration_days") == dur)
    p_arr = sub["prevalence"].to_numpy()
    d_arr = sub["delta_auc"].to_numpy()
    e_arr = sub["occurs_auc"].to_numpy()
    i_arr = sub["eic_auc"].to_numpy()

    r1, p1 = sp_stats.spearmanr(p_arr, d_arr)
    r2, p2 = sp_stats.spearmanr(p_arr, e_arr)
    r3, p3 = sp_stats.spearmanr(p_arr, i_arr)

    per_dur_rows.append({"duration": dur, "metric": "rho(prev, delta_auc)", "rho": round(r1, 3), "p_value": p1})
    per_dur_rows.append({"duration": dur, "metric": "rho(prev, EQ_auc)", "rho": round(r2, 3), "p_value": p2})
    per_dur_rows.append({"duration": dur, "metric": "rho(prev, EIC_auc)", "rho": round(r3, 3), "p_value": p3})

pl.DataFrame(per_dur_rows)